# 10年定着予測 - 探索的データ分析レポート v3（新規6分析）

**目的**: `data_exploration_report.md`（初回EDA）・`data_exploration_v2_report.md`（追加5分析）に続き、
これまで手つかずだった以下6つの観点を分析する。特徴量エンジニアリング（11_〜19_）で判明した知見を
踏まえた上で、**スコア向上を直接の目的とせず、新しい情報源を発掘すること**を優先する。

1. 昇進・降格の方向性（役割・等級変化の質的な向き）
2. 入社時メモの構造化パース（キャリア志向・転居許容・在宅希望などの定型カテゴリ抽出）
3. 離職の伝染・同調効果（部署内の休職・離職率）
4. 早期/中期/後期離職者の多変量プロファイル比較（残業時間以外の指標）
5. 前職経験と初期等級の整合性
6. 専攻分野×初期職種のマッチ/ミスマッチ

In [ ]:
import re
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")  # Colab想定。ローカル実行時は適宜変更
INPUT_DIR = PROJECT_ROOT / "data" / "input"

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
train_full = pd.read_csv(INPUT_DIR / "employee_monthly_train_full.csv")

y = train_persona[TARGET_COL]
print(f"Train Persona: {train_persona.shape}, Train Monthly: {train_monthly.shape}, Train Full: {train_full.shape}")
print(f"定着率: {y.mean():.4f}")

## 1. 昇進・降格の方向性

初回レポート8.8節で「等級上昇速度」が提案されていたが、これまで役割・等級の変化は**変化回数のみ**
特徴量化されており、上がったか下がったかという**方向性**は分析されていなかった。

まず「役割」に序列があるかを、対応する「等級」の分布から確認する。

In [ ]:
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_monthly["等級_num"] = train_monthly["等級"].map(grade_map)

role_grade = train_monthly.groupby("役割")["等級_num"].agg(["mean", "min", "max", "count"]).sort_values("mean")
print("役割ごとの等級分布（序列の確認）:")
print(role_grade)

役割は`メンバー(G1-2) < シニア(G2-3) < リード(G3-4) < エキスパート(G4) < シニアエキスパート/マネージャー(G5)`
という明確な序列を持つことが確認できた（G5では専門職トラック=シニアエキスパートと管理職トラック=マネージャーに分岐）。
これを踏まえ、役割にも序列番号を付与する。

In [ ]:
role_order = {"メンバー": 1, "シニア": 2, "リード": 3, "エキスパート": 4, "シニアエキスパート": 5, "マネージャー": 5}
train_monthly["役割_num"] = train_monthly["役割"].map(role_order)

promotion_records = []
for emp_id, emp_data in train_monthly.sort_values("経過月数").groupby("社員ID"):
    grades = emp_data["等級_num"].values
    roles = emp_data["役割_num"].values

    grade_diff = np.diff(grades)
    role_diff = np.diff(roles)

    promotion_records.append({
        "社員ID": emp_id,
        "等級_純増減": grades[-1] - grades[0],
        "等級_昇進回数": int((grade_diff > 0).sum()),
        "等級_降格回数": int((grade_diff < 0).sum()),
        "役割_純増減": roles[-1] - roles[0],
        "役割_昇進回数": int((role_diff > 0).sum()),
        "役割_降格回数": int((role_diff < 0).sum()),
    })

promo_df = pd.DataFrame(promotion_records).merge(train_persona[[ID_COL, TARGET_COL]], on=ID_COL)
print("0-23ヶ月における等級・役割の変化回数（重要な発見）:")
print(promo_df[["等級_純増減", "等級_昇進回数", "等級_降格回数", "役割_純増減", "役割_昇進回数", "役割_降格回数"]].sum())
print()
print(f"0-23ヶ月以内に等級/役割が変化した社員数: {((promo_df['等級_昇進回数']>0)|(promo_df['役割_昇進回数']>0)).sum()} / {len(promo_df)}")

**重要な発見**: 0-23ヶ月の観測窓内では、**全社員2,761名のうち誰一人として等級・役割の変化を
経験していない**（全員が入社時の等級・役割のまま）。これはバグではなく、データ生成上の設計であることを
`employee_monthly_train_full.csv`（10年分のフルデータ）で確認する。

In [ ]:
grade_map_full = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_full["等級_num"] = train_full["等級"].map(grade_map_full)

first_promotion_months = []
for emp_id, emp_data in train_full.sort_values("経過月数").groupby(ID_COL):
    grades = emp_data["等級_num"].values
    months = emp_data["経過月数"].values
    diff = np.diff(grades)
    idx = np.where(diff > 0)[0]
    if len(idx) > 0:
        first_promotion_months.append(months[idx[0] + 1])

first_promotion_months = np.array(first_promotion_months)
print(f"10年間で一度でも昇進した社員数: {len(first_promotion_months)} / {train_full[ID_COL].nunique()}")
print(f"初回昇進月の最小値: {first_promotion_months.min()}ヶ月")
print(pd.Series(first_promotion_months).describe())

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(first_promotion_months, bins=30)
ax.axvline(24, color="red", linestyle="--", label="24ヶ月（0-23ヶ月観測窓の境界）")
ax.set_title("初回昇進月の分布（10年間フルデータ）")
ax.set_xlabel("経過月数")
ax.legend()
plt.tight_layout()
plt.show()

### 考察（1）: 昇進・降格の方向性は0-23ヶ月データからは特徴量化できない

10年間のフルデータで確認したところ、**最初の昇進が発生するのは必ず24ヶ月目以降**であり、
0-23ヶ月以内に昇進・降格が発生した社員はゼロだった。これは「0-23ヶ月以内は昇進評価の対象外」という
制度設計（またはデータ生成ルール）を反映していると考えられる。

**結論**: Train/Testとも0-23ヶ月のデータしか利用できないため、**昇進・降格の方向性は原理的に
特徴量化できない**（分散がゼロで、モデルにとって定数列にしかならない）。これは「新規特徴量が
見つからなかった」という結果自体が、月次データの構造に関する重要な確認になった、という位置づけで
記録する。

## 2. 入社時メモの構造化パース

`入社時メモ`は「経歴／人物所見／キャリア志向／勤務地・働き方」という定型フォーマットを持つ。
これまでTF-IDF・文埋め込みという"ブラックボックス"表現でしか扱っていなかったが、
定型文の背後には少数のカテゴリがあることが分かっている（例:「キャリア志向：安定志向。」）。
正規表現でカテゴリを抽出し、定着率との関係を確認する。

In [ ]:
def extract_section(text, section_name):
    if pd.isna(text):
        return None
    m = re.search(rf"{section_name}：(.+?)(?:\n|$)", text)
    return m.group(1).strip() if m else None

train_persona["career_section"] = train_persona["入社時メモ"].apply(lambda t: extract_section(t, "キャリア志向"))

career_patterns = Counter(train_persona["career_section"].dropna())
print("キャリア志向セクションの文パターン(上位10):")
for pat, cnt in career_patterns.most_common(10):
    print(cnt, pat)

In [ ]:
def classify_career(s):
    if s is None:
        return "unknown"
    if "限定していない" in s or "限定しない" in s:
        return "未定"
    if "管理職" in s:
        return "管理職志向"
    if "専門職" in s:
        return "専門職志向"
    if "安定" in s:
        return "安定志向"
    return "other"

train_persona["career_cat"] = train_persona["career_section"].apply(classify_career)
career_rate = train_persona.groupby("career_cat")[TARGET_COL].agg(["mean", "count"]).sort_values("mean")
print(career_rate)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(career_rate.index, career_rate["mean"])
ax.set_title("キャリア志向カテゴリ 別 定着率")
ax.set_ylabel("定着率")
ax.axhline(y.mean(), color="red", linestyle="--", label=f"全体平均({y.mean():.3f})")
ax.legend()
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None

train_persona["ws_section"] = train_persona["入社時メモ"].apply(extract_workstyle_section)

NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")

def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None

NEG_REMOTE = re.compile(r"在宅勤務を(必須条件としていない|希望しない|希望していない|希望せず)")
POS_REMOTE = re.compile(r"在宅勤務を希望")

def classify_remote(s):
    if s is None:
        return None
    if POS_REMOTE.search(s):
        return True
    if NEG_REMOTE.search(s):
        return False
    return None

def extract_location(s):
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None

train_persona["転居許容"] = train_persona["ws_section"].apply(classify_reloc)
train_persona["在宅希望"] = train_persona["ws_section"].apply(classify_remote)
train_persona["希望勤務地"] = train_persona["ws_section"].apply(extract_location)

print("転居許容 別 定着率:")
print(train_persona.groupby("転居許容")[TARGET_COL].agg(["mean", "count"]))
print()
print("在宅希望 別 定着率:")
print(train_persona.groupby("在宅希望")[TARGET_COL].agg(["mean", "count"]))
print()
loc_rate = train_persona.groupby("希望勤務地")[TARGET_COL].agg(["mean", "count"])
loc_rate = loc_rate[loc_rate["count"] >= 20].sort_values("mean")
print("希望勤務地(n>=20) 別 定着率:")
print(loc_rate)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

reloc_rate = train_persona.groupby("転居許容")[TARGET_COL].mean()
axes[0].bar(reloc_rate.index.astype(str), reloc_rate.values)
axes[0].set_title("転居許容 別 定着率")

remote_rate = train_persona.groupby("在宅希望")[TARGET_COL].mean()
axes[1].bar(remote_rate.index.astype(str), remote_rate.values)
axes[1].set_title("在宅希望 別 定着率")

axes[2].bar(loc_rate.index, loc_rate["mean"])
axes[2].set_title("希望勤務地 別 定着率(n>=20)")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 考察（2）
キャリア志向・転居許容・在宅希望・希望勤務地のカテゴリ別定着率を確認した。差が大きいカテゴリがあれば、
これまでTF-IDF/文埋め込みという間接的な表現でしか捉えられていなかった情報を、直接的なカテゴリ変数として
補強できる可能性がある。

## 3. 離職の伝染・同調効果（部署内の休職・離職率）

「自分の初期部署で、他の同僚がどれだけ休職・離職しているか」という社会的な観点はこれまで
（部署IDのTarget Encoding以外は）分析されていない。0-23ヶ月のうちに`休職`または`退職`ステータスに
なった同僚の比率を部署ごとに集計し、自分自身の定着率との関係を確認する
（自分自身の休職/退職は分母・分子から除外し、あくまで「他者の状況」を見る）。

In [ ]:
status_by_emp = train_monthly.groupby(ID_COL)["月末在籍状態"].apply(lambda s: (s == "退職").any() or (s == "休職").any())
status_df = status_by_emp.reset_index()
status_df.columns = [ID_COL, "自身の休職退職経験"]

initial_dept = train_monthly[train_monthly["経過月数"] == 0][[ID_COL, "部署ID"]]
dept_status = initial_dept.merge(status_df, on=ID_COL)

dept_summary = dept_status.groupby("部署ID").agg(
    部署人数=("自身の休職退職経験", "size"),
    休職退職者数=("自身の休職退職経験", "sum"),
)
dept_summary["休職退職率"] = dept_summary["休職退職者数"] / dept_summary["部署人数"]
print(f"部署数: {len(dept_summary)}, うち人数5名以上: {(dept_summary['部署人数'] >= 5).sum()}")

def leave_one_out_rate(row_dept, row_self_flag, dept_summary):
    n = dept_summary.loc[row_dept, "部署人数"]
    cnt = dept_summary.loc[row_dept, "休職退職者数"]
    if row_self_flag:
        cnt -= 1
    n_others = n - 1
    return cnt / n_others if n_others > 0 else np.nan

dept_status["同僚休職退職率(自分除く)"] = dept_status.apply(
    lambda r: leave_one_out_rate(r["部署ID"], r["自身の休職退職経験"], dept_summary), axis=1
)

contagion_df = dept_status.merge(train_persona[[ID_COL, TARGET_COL]], on=ID_COL)
print(contagion_df["同僚休職退職率(自分除く)"].describe())

In [ ]:
contagion_df["同僚休職退職率_bin"] = pd.qcut(
    contagion_df["同僚休職退職率(自分除く)"], q=5, duplicates="drop"
)
bin_summary = contagion_df.groupby("同僚休職退職率_bin")[TARGET_COL].agg(["mean", "count"])
print(bin_summary)

corr = contagion_df["同僚休職退職率(自分除く)"].corr(contagion_df[TARGET_COL])
print(f"\n相関係数: {corr:.4f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(bin_summary.index.astype(str), bin_summary["mean"])
ax.set_title(f"初期部署の同僚休職退職率(自分除く) 別 定着率 (相関={corr:.3f})")
ax.set_ylabel("定着率")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 考察（3）
相関係数と5分位ビンごとの定着率の傾きから、部署内の同調・伝染効果がどの程度あるかを確認する。
なお、この特徴量を実際にモデルへ組み込む際は、部署IDのTarget Encoding同様、
学習期間のIDのみで計算しリークを防ぐ必要がある点に注意する
（`15_`/`16_`で発見した部署Target Encodingのリークと同じ構造の危険がある）。

## 4. 早期/中期/後期離職者の多変量プロファイル比較

`data_exploration_v2_report.md`の7.2節では0-23ヶ月の残業時間のみを4群（定着・早期離職12-23ヶ月・
中期離職24-59ヶ月・後期離職60-119ヶ月）で比較した。ここでは360度評価5項目・欠勤日数・月例給与・
有給取得日数・研修時間・上司面談回数・情報共有件数・顧客満足度・担当プロジェクト数についても
同様に比較し、離職タイミングによって「効く」指標が異なるかを確認する。

In [ ]:
def determine_group(retired_month):
    if pd.isna(retired_month):
        return "定着(10年)"
    if retired_month <= 23:
        return "早期離職(0-23ヶ月)"
    if retired_month <= 59:
        return "中期離職(24-59ヶ月)"
    return "後期離職(60-119ヶ月)"

retire_month = train_full[train_full["月末在籍状態"] == "退職"].groupby(ID_COL)["経過月数"].min()
group_df = train_persona[[ID_COL]].copy()
group_df["退職月"] = group_df[ID_COL].map(retire_month)
group_df["group"] = group_df["退職月"].apply(determine_group)

print(group_df["group"].value_counts())

In [ ]:
metrics = [
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度",
    "欠勤日数", "月例給与_円", "有給取得日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "顧客満足度評価", "担当プロジェクト数",
]

group_order = ["定着(10年)", "早期離職(0-23ヶ月)", "中期離職(24-59ヶ月)", "後期離職(60-119ヶ月)"]

profile_rows = []
for metric in metrics:
    emp_means = train_monthly.groupby(ID_COL)[metric].mean().reset_index()
    emp_means = emp_means.merge(group_df[[ID_COL, "group"]], on=ID_COL)
    row = {"指標": metric}
    for g in group_order:
        row[g] = emp_means[emp_means["group"] == g][metric].mean()
    profile_rows.append(row)

profile_df = pd.DataFrame(profile_rows).set_index("指標")
print(profile_df.round(3))

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()
for i, metric in enumerate(metrics):
    ax = axes[i]
    vals = profile_df.loc[metric, group_order]
    ax.bar(group_order, vals)
    ax.set_title(metric, fontsize=10)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
for j in range(len(metrics), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

### 考察（4）
0-23ヶ月の指標のうち、定着者と後期離職者(60-119ヶ月)でほぼ差がない指標が多ければ、
`data_exploration_v2_report.md`7.2節の知見（0-23ヶ月データは早期離職の検出には強いが後期離職には
ほぼ効かない）が残業時間以外にも一般化することを意味する。逆に、後期離職者だけが明確に異なる指標が
見つかれば、それは新規特徴量の有力な候補になる。

## 5. 前職経験と初期等級の整合性

`data_exploration_v2_report.md`8節では初任給と初期等級の整合性を確認したが、
**中途入社者の前職経験月数と初期等級の整合性**はまだ確認されていない。
経験の割に等級が低い（または高い）ミスマッチが定着率に影響するかを見る。

In [ ]:
mid_career = train_persona[train_persona["入社区分"] == "中途"].copy()
mid_career["初期等級_num"] = mid_career["初期等級"].map(grade_map)

corr = mid_career["前職経験月数"].corr(mid_career["初期等級_num"])
print(f"中途入社者(n={len(mid_career)})の 前職経験月数 と 初期等級 の相関: {corr:.3f}")

exp_grade_mean = mid_career.groupby("初期等級")["前職経験月数"].agg(["mean", "std", "count"])
print(exp_grade_mean)

In [ ]:
from sklearn.linear_model import LinearRegression

X = mid_career[["前職経験月数"]].values
yy = mid_career["初期等級_num"].values
reg = LinearRegression().fit(X, yy)
mid_career["等級_予測残差"] = yy - reg.predict(X)

mid_career["整合性区分"] = pd.qcut(mid_career["等級_予測残差"], q=3, labels=["経験に対し等級が低い", "整合的", "経験に対し等級が高い"])
consistency_rate = mid_career.groupby("整合性区分")[TARGET_COL].agg(["mean", "count"])
print(consistency_rate)

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(consistency_rate.index.astype(str), consistency_rate["mean"])
ax.set_title("前職経験に対する初期等級の整合性 別 定着率(中途入社者)")
ax.set_ylabel("定着率")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 考察（5）
中途入社者について、前職経験の割に等級が低い/高いグループ間で定着率に差があるかを確認した。
差が明確であれば、「経験と等級の整合性残差」は新規特徴量の候補になる（初任給の等級内偏差と
似た発想だが、対象が前職経験という異なる軸である点が新しい）。

## 6. 専攻分野×初期職種のマッチ/ミスマッチ

専攻分野と配属された初期職種が「合っているか」を確認する。まずクロス集計で各専攻分野に対する
最頻出の初期職種を確認し、それを「マッチ」と定義する。

In [ ]:
ct = pd.crosstab(train_persona["専攻分野"], train_persona["初期職種"], normalize="index")
print(ct.round(3))

major_to_job = ct.idxmax(axis=1)
print("\n専攻分野ごとの最頻出初期職種（マッチの定義）:")
print(major_to_job)

In [ ]:
train_persona["専攻職種マッチ"] = train_persona.apply(
    lambda r: r["初期職種"] == major_to_job.get(r["専攻分野"]), axis=1
)
match_rate = train_persona.groupby("専攻職種マッチ")[TARGET_COL].agg(["mean", "count"])
print(match_rate)

# カイ二乗検定
from scipy.stats import chi2_contingency
ct2 = pd.crosstab(train_persona["専攻職種マッチ"], train_persona[TARGET_COL])
chi2, p, dof, expected = chi2_contingency(ct2)
print(f"\nカイ二乗検定: chi2={chi2:.3f}, p={p:.4f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(match_rate.index.astype(str), match_rate["mean"])
ax.set_title(f"専攻×職種マッチ 別 定着率 (カイ二乗検定 p={p:.3f})")
ax.set_ylabel("定着率")
ax.axhline(y.mean(), color="red", linestyle="--")
plt.tight_layout()
plt.show()

### 考察（6）
専攻分野と初期職種の「最頻出マッチ」を基準にした場合の定着率の差を確認した。クロス集計自体
（本セクション冒頭）を見ると、どの専攻も特定の職種に極端に偏ってはおらず、緩やかな相関にとどまる
ことが分かる。そのため、この特徴量の効果は他の5分析（特にキャリア志向・部署の同調効果）と比べて
限定的である可能性がある。

## 7. 自己学習（詳細）の内容分析（新発見）

`data_input/employee_dataset_specification.md`を再確認したところ、月次データの**`自己学習（詳細）`**列
（例:「クラウド基盤入門：2.0時間｜統計学基礎：1.5時間」、学習がない月は「受講なし」）が、
これまでの特徴量エンジニアリング（`11_`〜`19_`）でもEDA（v1・v2・本v3の1-6節）でも
**一度も使われていなかった**ことが判明した。新規の情報源として分析する。

In [ ]:
def parse_all_themes(s):
    if s == "受講なし":
        return [], 0.0
    parts = s.split("｜")
    themes, total = [], 0.0
    for part in parts:
        m2 = re.match(r"(.+?)：([\d.]+)時間", part)
        if m2:
            themes.append(m2.group(1))
            total += float(m2.group(2))
    return themes, total

parsed = train_monthly["自己学習（詳細）"].apply(parse_all_themes)
train_monthly["自己学習テーマ数"] = parsed.apply(lambda x: len(x[0]))
train_monthly["自己学習時間"] = parsed.apply(lambda x: x[1])

all_themes = set(t for tl, _ in parsed for t in tl)
print(f"ユニークテーマ数: {len(all_themes)}")
print(f"相関(自己学習時間 vs 研修時間): {train_monthly['自己学習時間'].corr(train_monthly['研修時間']):.4f}")
print("→ 既存の「研修時間」（会社主導の研修）とはほぼ無相関で、独立した情報源であることを確認。")

既存の「研修時間」（会社主導の研修）とはほぼ無相関（相関0.004程度）であり、独立した情報源であることが
確認できた。次に、社員ごとに0-23ヶ月の自己学習量を集計し、定着率との関係を見る。

In [ ]:
emp_total_hours = train_monthly.groupby(ID_COL)["自己学習時間"].sum()
emp_active_months = train_monthly[train_monthly["自己学習時間"] > 0].groupby(ID_COL).size()
emp_unique_themes = train_monthly.groupby(ID_COL).apply(
    lambda g: len(set(t for tl in parsed.loc[g.index].apply(lambda x: x[0]) for t in tl))
)

study_df = pd.DataFrame({
    "自己学習合計時間": emp_total_hours,
    "自己学習実施月数": emp_active_months,
    "自己学習ユニークテーマ数": emp_unique_themes,
}).fillna(0).reset_index()
study_df = study_df.merge(train_persona[[ID_COL, TARGET_COL, "初期職種"]], on=ID_COL)

for col in ["自己学習合計時間", "自己学習実施月数", "自己学習ユニークテーマ数"]:
    corr = study_df[col].corr(study_df[TARGET_COL])
    print(f"相関({col}, 定着ラベル): {corr:.4f}")

In [ ]:
study_df["実施月数_bin"] = pd.qcut(study_df["自己学習実施月数"], q=5, duplicates="drop")
bin_rate = study_df.groupby("実施月数_bin", observed=True)[TARGET_COL].agg(["mean", "count"])
print(bin_rate)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(bin_rate.index.astype(str), bin_rate["mean"])
ax.axhline(y.mean(), color="red", linestyle="--", label=f"全体平均({y.mean():.3f})")
ax.set_title("自己学習実施月数(5分位) 別 定着率")
ax.set_ylabel("定着率")
ax.legend()
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

自己学習実施月数の最下位分位（0-23ヶ月のうち自己学習をほとんどしなかった層）で定着率が明確に低い
傾向が見られた。次に、学習テーマの内容が本人の初期職種と合っているか（「テーマ職種マッチ度」）を確認する。
35種類のテーマを、6つの初期職種カテゴリに近い6分類へ人手でマッピングした。

In [ ]:
THEME_TO_JOB = {
    "クラウド基盤入門": "IT・エンジニアリング", "Pythonプログラミング基礎": "IT・エンジニアリング",
    "SQLによるデータ操作": "IT・エンジニアリング", "RPA基礎": "IT・エンジニアリング",
    "システム設計基礎": "IT・エンジニアリング", "アジャイル開発実践": "IT・エンジニアリング",
    "情報セキュリティ基礎": "IT・エンジニアリング", "機械学習入門": "IT・エンジニアリング",
    "Pythonによるデータ分析": "データ・商品企画・コンサルティング", "データ可視化実践": "データ・商品企画・コンサルティング",
    "統計学基礎": "データ・商品企画・コンサルティング", "BIダッシュボード作成": "データ・商品企画・コンサルティング",
    "データリテラシー基礎": "データ・商品企画・コンサルティング", "ロジカルシンキング実践": "データ・商品企画・コンサルティング",
    "人事データ分析入門": "データ・商品企画・コンサルティング",
    "金融リスク管理": "リスク・金融・コンプライアンス", "オペレーショナルリスク基礎": "リスク・金融・コンプライアンス",
    "コンプライアンス実務": "リスク・金融・コンプライアンス", "AML/CFT基礎": "リスク・金融・コンプライアンス",
    "内部統制入門": "リスク・金融・コンプライアンス", "財務会計基礎": "リスク・金融・コンプライアンス",
    "CRM活用基礎": "営業・顧客対応", "アカウントプランニング": "営業・顧客対応", "提案書作成実践": "営業・顧客対応",
    "顧客課題ヒアリング": "営業・顧客対応", "プレゼンテーション設計": "営業・顧客対応", "交渉・合意形成": "営業・顧客対応",
    "労務管理基礎": "コーポレート", "組織開発・人材育成": "コーポレート", "管理会計基礎": "コーポレート", "経営管理基礎": "コーポレート",
    "業務改善の進め方": "業務運用", "業務プロセス可視化": "業務運用", "品質管理入門": "業務運用", "プロジェクトマネジメント入門": "業務運用",
}
print(f"マッピング済みテーマ数: {len(THEME_TO_JOB)} / {len(all_themes)}")

from collections import Counter

def most_common_category(theme_lists):
    cats = [THEME_TO_JOB.get(t, "unknown") for tl in theme_lists for t in tl]
    return Counter(cats).most_common(1)[0][0] if cats else None

emp_main_cat = train_monthly.groupby(ID_COL).apply(
    lambda g: most_common_category(parsed.loc[g.index].apply(lambda x: x[0]))
)
emp_main_cat.name = "自己学習主カテゴリ"

match_df = emp_main_cat.reset_index().merge(train_persona[[ID_COL, "初期職種", TARGET_COL]], on=ID_COL)
match_df["テーマ職種マッチ"] = match_df["自己学習主カテゴリ"] == match_df["初期職種"]
print(f"\n自己学習が全くない社員: {match_df['自己学習主カテゴリ'].isna().sum()}名")
print(match_df.groupby("テーマ職種マッチ")[TARGET_COL].agg(["mean", "count"]))

from scipy.stats import chi2_contingency
ct = pd.crosstab(match_df["テーマ職種マッチ"], match_df[TARGET_COL])
chi2, pval, _, _ = chi2_contingency(ct)
print(f"\nカイ二乗検定 p値: {pval:.4f}")

### 考察（7）

- **自己学習実施月数**は0-23ヶ月データの中でも新しい情報源であり（既存の「研修時間」とは無相関）、
  実施月数が少ない層ほど定着率が低い傾向が確認できた。合計時間・ユニークテーマ数も弱い相関を示した。
- 一方、**学習テーマの内容が初期職種と合っているか（テーマ職種マッチ度）はカイ二乗検定で有意差が
  出ず**、専攻分野×初期職種のマッチ（本レポート6節）と同様に効果は限定的だった。
- **結論**: 「何を学んだか」よりも「継続して自己学習に取り組んだか（実施月数）」の方が定着との
  関連が強い。これは、自己学習という行動そのものが仕事への関与度・自己成長意欲の代理指標になっている
  可能性を示唆する。新規特徴量としては、`自己学習実施月数`・`自己学習合計時間`・`自己学習ユニークテーマ数`
  を優先し、テーマ職種マッチ度は優先度を下げる。

## 8. 総合考察と特徴量エンジニアリングへの示唆

7つの分析結果を踏まえ、新規特徴量として実装する価値が高いと考えられる候補を整理する
（実際の採否は、既存の教訓——「特徴量重要度が高い＝汎化に貢献するとは限らないため単体アブレーションで
検証すべき」（`16_`）・「小規模グループのTarget Encodingは学習期間のIDのみでfitしないとリークする」
（`15_`）——を踏まえて次の特徴量エンジニアリングノートブックで検証する）。

| # | 分析 | 新規特徴量の候補 | 実装上の注意 |
|---|---|---|---|
| 1 | 昇進・降格の方向性 | **実装不可（採用見送り）** | 0-23ヶ月以内は昇進・降格が発生しないため、分散ゼロで特徴量化できないことが判明 |
| 2 | 入社時メモの構造化パース | キャリア志向カテゴリ、転居許容、在宅希望、希望勤務地 | 抽出漏れ（未知パターン）は欠損値として扱う |
| 3 | 部署内の同調・伝染効果 | 初期部署の同僚休職退職率(自分除く) | 目的変数由来の情報のため、部署Target Encodingと同様、**学習期間のIDのみでfitする必要がある**（`15_`/`16_`のリーク教訓が直接当てはまる） |
| 4 | 離職タイミング別プロファイル | 差が明確だった指標があれば、その指標を追加/強調 | 0-23ヶ月データの本質的な限界（後期離職の判別困難）を再確認する目的が主 |
| 5 | 前職経験と等級の整合性 | 経験・等級の整合性残差（中途入社者のみ） | 新卒には適用できないため、フラグ列と併用が必要 |
| 6 | 専攻×職種マッチ | マッチフラグ | クロス集計の結果次第では効果が薄い可能性が高い |
| 7 | 自己学習（詳細）の内容分析 | 自己学習実施月数、合計時間、ユニークテーマ数 | 既存の研修時間とは独立した新規の情報源。テーマ職種マッチ度は効果薄 |

**次のアクション**: 上記のうち少なくとも**2・5・7**を新しい特徴量エンジニアリングノートブック
（`20_`想定）に実装し、単体アブレーション（`16_`と同じ枠組み）で効果を検証する。1（昇進・降格）は
構造的に実装不可と判明したため見送り、3・6は効果が限定的な可能性が高いため優先度を下げる。
ユーザーの意向（スコアが下がっても新規性を優先）を踏まえ、Publicスコアの改善有無にかかわらず、
これらの特徴量が実際にどう振る舞うかを記録することを優先する。